In [27]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pandas as pd

nltk.download('vader_lexicon')

df = pd.read_csv("spotify_millsongdata.csv")

additional_df = pd.read_csv("additional_features.csv")

vds = SentimentIntensityAnalyzer()

records = []
for _, row in df.iterrows():
    scores = vds.polarity_scores(str(row["text"]))
    records.append({
        "Song":     row["song"],
        "Artist":   row["artist"],
        "Compound": scores["compound"],
        "Negative":      scores["neg"],
        "Neutral":      scores["neu"],
        "Positive":      scores["pos"],
    })

sentiment_df = pd.DataFrame(records)

merged_df = pd.merge(sentiment_df, additional_df, how="inner", left_on="Song", right_on="track_name")

# Drop duplicate song name column if needed
merged_df.drop(columns=["Unnamed: 0", "track_id", "duration_ms", "time_signature", "track_name", "artists", "album_name"], inplace=True)
merged_df = merged_df.drop_duplicates(subset=['Song'], keep='first')

# Save merged dataset to CSV
merged_df.to_csv("merged_spotify_dataset.csv", index=False)

print("Merged dataset saved as 'merged_spotify_dataset.csv'")

print(merged_df.columns)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/linhha/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Merged dataset saved as 'merged_spotify_dataset.csv'
Index(['Song', 'Artist', 'Compound', 'Negative', 'Neutral', 'Positive',
       'popularity', 'explicit', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'track_genre'],
      dtype='object')


In [ ]:
import torch
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

merged_df = pd.read_csv("merged_spotify_dataset.csv") 

#initialize embedder and regressor
device    = "cuda" if torch.cuda.is_available() else "cpu"
embedder  = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)
regressor = Ridge()

#build text descriptions for each track
def row_to_description(row):
    return (f"A {row['track_genre']} song called '{row['Song']}' by {row['Artist']} "
            f"that is {'explicit' if row['explicit'] else 'not explicit'}, "
            f"with popularity {row['popularity']:.0f}, "
            f"{'high' if row['energy']>0.6 else 'low'} energy, "
            f"and {'high' if row['acousticness']>0.6 else 'low'} acousticness.")

descriptions = merged_df.apply(row_to_description, axis=1).tolist()

#compute embeddings for all tracks (only once)
X_embeddings = embedder.encode(
    descriptions,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

#train a Ridge regressor on true Spotify features
feature_cols = [
    'Compound','Negative','Neutral','Positive',
    'danceability','energy','key','loudness',
    'mode','speechiness','acousticness',
    'instrumentalness','liveness','valence','tempo'
]
y = merged_df[feature_cols].values

X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings, y, test_size=0.2, random_state=42
)
regressor.fit(X_train, y_train)
print(f"R² on test set: {regressor.score(X_test, y_test):.3f}")

#kmeans on embeddings
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, random_state=42).fit(X_embeddings)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

R² on test set: 0.236


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/linhha/opt/anaconda3/lib/python3.9/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


In [ ]:
#dynamic query processing

import numpy as np
from sentence_transformers import SentenceTransformer

def process_user_query(query_text, k=10):
    query_emb = embedder.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    pred_feats = regressor.predict(query_emb)[0]
    print("Predicted feature vector:")
    for name, val in zip(feature_cols, pred_feats):
        print(f"  {name}: {val:.3f}")
    
    cluster_label = kmeans.predict(query_emb)[0]
    print(f"\nAssigned to cluster: {cluster_label}")
    
    cluster_idxs = np.where(kmeans.labels_ == cluster_label)[0]
    dists = np.linalg.norm(X_embeddings[cluster_idxs] - query_emb, axis=1)
    nearest = cluster_idxs[np.argsort(dists)[:k]]
    neighbors = merged_df.iloc[nearest][['Song','Artist']].reset_index(drop=True)
    
    print(f"\nTop {k} nearest songs in the same cluster:")
    print(neighbors)

while True:
    q = input("Describe the kind of music you're in the mood for: (enter 'quit' to exit)")
    if q.lower() == 'quit':
        break
    process_user_query(q, k=10)


Predicted feature vector:
  Compound: 1.031
  Negative: 0.038
  Neutral: 0.779
  Positive: 0.183
  danceability: 0.603
  energy: 0.572
  key: 6.023
  loudness: -11.325
  mode: 0.861
  speechiness: 0.097
  acousticness: 0.592
  instrumentalness: 0.333
  liveness: 0.239
  valence: 0.423
  tempo: 122.642

Assigned to cluster: 4

Top 10 nearest songs in the same cluster:
              Song                 Artist
0  All Summer Long             Beach Boys
1      Summer Days            Ace Of Base
2       Daydreamer                  Adele
3           Shiver               Coldplay
4             Snow  Red Hot Chili Peppers
5      I Feel Good               Boney M.
6           Sparks               Coldplay
7          Take Me           Donna Summer
8             Open            Demi Lovato
9       Cloudburst                  Oasis
Predicted feature vector:
  Compound: 0.913
  Negative: 0.057
  Neutral: 0.757
  Positive: 0.187
  danceability: 0.640
  energy: 0.504
  key: 5.985
  loudness: -12.303
